<a href="https://colab.research.google.com/github/Sahar-Hassan5588/Assignment2-Advanced-Topics/blob/main/assignment2.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Install the required libraries

In [ ]:
!pip install -q transformers accelerate sentencepiece pandas scipy matplotlib huggingface_hub

# Import the libraries

In [ ]:
import pandas as pd
import numpy as np
import json
import re
import torch
from transformers import AutoTokenizer, AutoModelForCausalLM
from huggingface_hub import login
from scipy import stats
import matplotlib.pyplot as plt

# Login to Hugging Face (required for Llama 3)

In [ ]:
login()

/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


# Load google/gemma model


In [ ]:
model_name = "google/gemma-2b-it"

print("Loading model:", model_name)

tokenizer = AutoTokenizer.from_pretrained(model_name)

model = AutoModelForCausalLM.from_pretrained(
    model_name,
    dtype=torch.float16,        # updated (no warning)
    device_map="auto"
)

print("Model loaded successfully")

Loading model: google/gemma-2b-it


config.json:   0%|          | 0.00/627 [00:00<?, ?B/s]

tokenizer_config.json: 0.00B [00:00, ?B/s]

tokenizer.json:   0%|          | 0.00/17.5M [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/636 [00:00<?, ?B/s]

model.safetensors.index.json: 0.00B [00:00, ?B/s]

Fetching 2 files:   0%|          | 0/2 [00:00<?, ?it/s]

Loading weights:   0%|          | 0/164 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/137 [00:00<?, ?B/s]

Model loaded successfully


# Generation function for Gemma:
This function sends a prompt to the model and returns only the model’s answer.

In [ ]:
def generate_response(prompt):
    messages = [
        {"role": "user", "content": prompt}
    ]

    inputs = tokenizer.apply_chat_template(
        messages,
        tokenize=True,
        add_generation_prompt=True,
        return_tensors="pt"
    )

    input_ids = inputs["input_ids"].to(model.device)
    attention_mask = inputs["attention_mask"].to(model.device)

    outputs = model.generate(
        input_ids=input_ids,
        attention_mask=attention_mask,
        max_new_tokens=600,
        temperature=0.3,
        do_sample=True
    )

    generated_tokens = outputs[0][input_ids.shape[-1]:]
    response = tokenizer.decode(generated_tokens, skip_special_tokens=True)

    return response

#  Prompt 1 (JSON version)

In [ ]:
PROMPT_1_FINAL = """I want to make three personas, and the three agents. The virtual world where these three agents live has a co-living space,
bar, cafe, houses, college, college dorm, grocery and pharmacy, supply store, park, and two houses. Can you create personas of all
three agents for me? I want you to provide me, with their Age, Educational Qualification, Personality Traits, Devices and technologies
they use, Work experience, Domain of work, Country, Gender with the following requirements:

Requirements:
• Names (mandatory): Ensure the names reflect a variety of ethnicities and faiths.
• Gender (mandatory): Include a balanced representation of different genders.
• Age (mandatory): Include a balanced representation of different genders.
• Personality Traits (mandatory): Include diverse personality traits
• Domain of Work (mandatory): Focus on diverse roles.
• Geographical Location (mandatory): Represent various regions globally.
• Education level (mandatory): No education, High school, Undergraduate, Master’s , phD.
• Years of experience (mandatory): Junior/Beginner, Mid, Senior.
• Character Limit (optional): Each profile must be concise, within 300 characters.

IMPORTANT:
Return ONLY valid JSON in this exact structure:

{
  "personas": [
    {
      "id": "P1",
      "name": "",
      "age": "",
      "gender": "",
      "education_level": "",
      "personality_traits": [],
      "devices": [],
      "experience_level": "",
      "domain_of_work": "",
      "country": ""
    },
    {
      "id": "P2",
      "name": "",
      "age": "",
      "gender": "",
      "education_level": "",
      "personality_traits": [],
      "devices": [],
      "experience_level": "",
      "domain_of_work": "",
      "country": ""
    },
    {
      "id": "P3",
      "name": "",
      "age": "",
      "gender": "",
      "education_level": "",
      "personality_traits": [],
      "devices": [],
      "experience_level": "",
      "domain_of_work": "",
      "country": ""
    }
  ]
}

Rules:
- age must be a number only
- gender must be one of: Male, Female
- domain_of_work must be a short category
- country must be country only
- personality_traits must be a short list
- do not add any text outside JSON
"""

# Run Prompt 1

In [ ]:
response = generate_response(PROMPT_1_FINAL)
print(response)

{
  "personas": [
    {
      "id": "P1",
      "name": "Aisha",
      "age": 25,
      "gender": "Female",
      "education_level": "Master's",
      "personality_traits": ["Determined, Creative, Ambitious"],
      "devices": ["Laptop, Smartphone"],
      "experience_level": "Senior",
      "domain_of_work": "Software Engineering",
      "country": "United States"
    },
    {
      "id": "P2",
      "name": "John",
      "age": 32,
      "gender": "Male",
      "education_level": "Bachelor's",
      "personality_traits": ["Friendly, Social, Outgoing"],
      "devices": ["Phone, Headset"],
      "experience_level": "Mid",
      "domain_of_work": "Hospitality",
      "country": "Canada"
    },
    {
      "id": "P3",
      "name": "Maria",
      "age": 40,
      "gender": "Female",
      "education_level": "High School",
      "personality_traits": ["Independent, Resourceful, Hardworking"],
      "devices": ["Book, Laptop"],
      "experience_level": "Junior",
      "domain_of_work": "

In [ ]:
def extract_json(text):
    match = re.search(r'\{.*\}', text, re.DOTALL)
    if not match:
        raise ValueError("No valid JSON found in model output")
    return json.loads(match.group(0))

# Convert the model’s JSON text into usable data

In [ ]:
import json

persona_data = extract_json(response)
persona_data


{'personas': [{'id': 'P1',
   'name': 'Aisha',
   'age': 25,
   'gender': 'Female',
   'education_level': "Master's",
   'personality_traits': ['Determined, Creative, Ambitious'],
   'devices': ['Laptop, Smartphone'],
   'experience_level': 'Senior',
   'domain_of_work': 'Software Engineering',
   'country': 'United States'},
  {'id': 'P2',
   'name': 'John',
   'age': 32,
   'gender': 'Male',
   'education_level': "Bachelor's",
   'personality_traits': ['Friendly, Social, Outgoing'],
   'devices': ['Phone, Headset'],
   'experience_level': 'Mid',
   'domain_of_work': 'Hospitality',
   'country': 'Canada'},
  {'id': 'P3',
   'name': 'Maria',
   'age': 40,
   'gender': 'Female',
   'education_level': 'High School',
   'personality_traits': ['Independent, Resourceful, Hardworking'],
   'devices': ['Book, Laptop'],
   'experience_level': 'Junior',
   'domain_of_work': 'Nursing',
   'country': 'Brazil'}]}

# Create Prompt 2

In [ ]:
PROMPT_2 = """Among these three agents, if you were to make one of them more vulnerable to phishing, who would you choose and why?
Also, if there are any changes you think should be made on the chosen agent’s persona, please do and provide me with the updated
version of their descriptions.

IMPORTANT:
Return ONLY valid JSON in this format:

{
  "selected_persona": "",
  "reason": "",
  "updated_persona": ""

}

Rules:
- Select ONLY ONE persona.
- Do not add extra text.

"""

# Combine personas with Prompt 2

In [ ]:
prompt2_input = PROMPT_2 + "\n\nHere are the three personas:\n" + json.dumps(persona_data, indent=2)
print(prompt2_input)

Among these three agents, if you were to make one of them more vulnerable to phishing, who would you choose and why?
Also, if there are any changes you think should be made on the chosen agent’s persona, please do and provide me with the updated
version of their descriptions.

IMPORTANT:
Return ONLY valid JSON in this format:

{
  "selected_persona": "",
  "reason": "",
  "updated_persona": ""

}

Rules:
- Select ONLY ONE persona.
- Do not add extra text.



Here are the three personas:
{
  "personas": [
    {
      "id": "P1",
      "name": "Aisha",
      "age": 25,
      "gender": "Female",
      "education_level": "Master's",
      "personality_traits": [
        "Determined, Creative, Ambitious"
      ],
      "devices": [
        "Laptop, Smartphone"
      ],
      "experience_level": "Senior",
      "domain_of_work": "Software Engineering",
      "country": "United States"
    },
    {
      "id": "P2",
      "name": "John",
      "age": 32,
      "gender": "Male",
      "educati

# Run Prompt 2

In [ ]:
prompt2_output = generate_response(prompt2_input)
print(prompt2_output)

**Selected Persona:** P1

**Reason:** Aisha is the persona with the most sensitive information and activities. Her devices contain sensitive personal and professional data, and she has a high exposure to phishing attacks due to her professional nature and use of multiple devices.

**Updated Persona Description:**

```json
{
  "id": "P1",
  "name": "Aisha",
  "age": 25,
  "gender": "Female",
  "education_level": "Master's",
  "personality_traits": [
    "Determined, Creative, Ambitious"
  ],
  "devices": [
    {
      "type": "Laptop",
      "operating_system": "Windows 10",
      "data_storage_capacity": "512GB"
    },
    {
      "type": "Smartphone",
      "operating_system": "Android 11",
      "data_storage_capacity": "128GB"
    }
  ],
  "experience_level": "Senior",
  "domain_of_work": "Software Engineering",
  "country": "United States"
}
```


In [ ]:
def parse_prompt2_output(text):
    try:
        return extract_json(text)
    except Exception:
        persona_match = re.search(r'\{[^{}]*"id"\s*:\s*"P[123]".*\}', text, re.DOTALL)
        if persona_match:
            persona_obj = json.loads(persona_match.group(0))
            return {
                "selected_persona": persona_obj.get("id", ""),
                "reason": "",
                "update_needed": "",
                "fields_changed": [],
                "updated_persona": persona_obj
            }
        raise ValueError("Could not parse Prompt 2 output")

In [ ]:
prompt2_data = parse_prompt2_output(prompt2_output)
prompt2_data

{'id': 'P1',
 'name': 'Aisha',
 'age': 25,
 'gender': 'Female',
 'education_level': "Master's",
 'personality_traits': ['Determined, Creative, Ambitious'],
 'devices': [{'type': 'Laptop',
   'operating_system': 'Windows 10',
   'data_storage_capacity': '512GB'},
  {'type': 'Smartphone',
   'operating_system': 'Android 11',
   'data_storage_capacity': '128GB'}],
 'experience_level': 'Senior',
 'domain_of_work': 'Software Engineering',
 'country': 'United States'}

In [ ]:
columns = [
    "Model",
    "Prompt1_Set_ID",
    "Prompt2_Run_ID",
    "Persona ID",
    "Persona Name",
    "Profile details",
    "Name",
    "Age",
    "Gender",
    "Personality Traits",
    "Domain of work",
    "Years of Exp.",
    "Location",
    "Education Level",
    "Devices and technologies use",
    "Reason(s)",
    "Y/N",
    "Raw Prompt 1 Output",
    "Raw Prompt 2 Output",
    "Interpretation 1"

]

df = pd.DataFrame(columns=columns)
df.head()

,Model,Prompt1_Set_ID,Prompt2_Run_ID,Persona ID,Persona Name,Profile details,Name,Age,Gender,Personality Traits,Domain of work,Years of Exp.,Location,Education Level,Devices and technologies use,Reason(s),Y/N,Raw Prompt 1 Output,Raw Prompt 2 Output,Interpretation 1


In [ ]:
import json

notebook_path = "/content/Assignment2-Advanced-Topics/Assignment2.ipynb"

with open(notebook_path, "r", encoding="utf-8") as f:
    nb = json.load(f)

# Remove broken widget metadata
if "metadata" in nb and "widgets" in nb["metadata"]:
    del nb["metadata"]["widgets"]

with open(notebook_path, "w", encoding="utf-8") as f:
    json.dump(nb, f, ensure_ascii=False, indent=1)

print("Cleaned notebook saved.")

FileNotFoundError: [Errno 2] No such file or directory: '/content/Assignment2-Advanced-Topics/Assignment2.ipynb'